In [52]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers peft


### CELL 1: SESSION TIME, HF SECRET AUTHENTICATION & DIAGNOSTICS


In [53]:

import os
import time
from transformers import utils as tf_utils
from huggingface_hub import login

# Enable verbose Hugging Face progress logs so download bars appear in real time
tf_utils.logging.set_verbosity_info()

def check_kaggle_uptime(limit_hours=9.0):
    """Reads container uptime directly from the Kaggle OS."""
    try:
        with open('/proc/uptime', 'r') as f:
            uptime_seconds = float(f.readline().split()[0])
        consumed_hours = uptime_seconds / 3600
        remaining_hours = limit_hours - consumed_hours
        print("========================================")
        print(f"⏱️ Kaggle Session Uptime : {consumed_hours:.2f} Hours ({uptime_seconds/60:.1f} Mins)")
        print(f"⏳ Remaining GPU Quota   : {remaining_hours:.2f} Hours ({remaining_hours*60:.1f} Mins)")
        print("========================================")
        if remaining_hours < 1.0:
            print("⚠️ WARNING: Less than 1 hour remaining on GPU session!")
    except Exception as e:
        print(f"Could not read system uptime: {e}")

check_kaggle_uptime()

# Retrieve Hugging Face Access Token automatically from Kaggle Secrets
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    try:
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = user_secrets.get_secret("HF-TOKEN")

    print("🔑 Kaggle Secret Found! Authenticating with Hugging Face...")
    login(token=hf_token)
    print("✅ Hugging Face Authentication Successful!")
except Exception as e:
    print(f"⚠️ Kaggle Secrets Retrieval Warning: {e}")
    print("👉 Ensure you added your token in Kaggle via: Add-ons -> Secrets -> Label: HF_TOKEN")

⏱️ Kaggle Session Uptime : 1.43 Hours (85.8 Mins)
⏳ Remaining GPU Quota   : 7.57 Hours (454.2 Mins)
🔑 Kaggle Secret Found! Authenticating with Hugging Face...
✅ Hugging Face Authentication Successful!


### CELL 2: CORE DEPENDENCIES & IMPORTS

In [54]:

print("\n📦 Loading Core Libraries...")
import gc
import json
import tarfile
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ==============================================================================
# CELL 3: UNIFIED PROMPT TEMPLATE FORMATTER
# ==============================================================================
def format_llama3_prompt(question_text, answer_text=None):
    """
    Wraps clinical questions in official LLaMA-3.1 Instruct format.
    The <image> token acts as a placeholder for 3D visual embeddings.
    """
    system_prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are an expert medical AI assistant analyzing 3D Brain MRI scans. "
        "Answer the user's clinical question accurately based on the provided visual embeddings.<|eot_id|>"
    )

    user_prompt = (
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"<image>\n{question_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

    full_prompt = system_prompt + user_prompt
    if answer_text:
        full_prompt += f"{answer_text}<|eot_id|>"

    return full_prompt

print("✅ Prompt Formatter Loaded.")

# ==============================================================================
# CELL 4: ZERO-DISK VIRTUAL CATALOG STREAMER & 3D MRI READER
# ==============================================================================
class BraTSVirtualStreamer:
    def __init__(self, tar_path="/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"):
        self.tar_path = tar_path
        self.temp_dir = "/tmp/brats_stream"
        os.makedirs(self.temp_dir, exist_ok=True)

    def stream_patient(self, patient_id):
        """Extracts a single patient to RAM/tmp, yields path, and auto-purges."""
        try:
            if os.path.exists(self.tar_path):
                with tarfile.open(self.tar_path, 'r') as tar:
                    patient_files = [m for m in tar.getmembers() if patient_id in m.name]
                    tar.extractall(path=self.temp_dir, members=patient_files)
                yield os.path.join(self.temp_dir, patient_id)
            else:
                # Fallback directory if path is uncompressed
                yield self.temp_dir
        finally:
            # Immediate Cleanup to keep disk usage at 0 MB
            for f in os.listdir(self.temp_dir):
                file_path = os.path.join(self.temp_dir, f)
                if os.path.isfile(file_path):
                    os.remove(file_path)

def load_patient_3d_volume(patient_dir):
    """
    Loads preprocessed 3D MRI volume files (.nii.gz or .npy) and converts to Tensor.
    """
    if not os.path.exists(patient_dir) or not os.listdir(patient_dir):
        # Create a fallback standard 3D volume shape if patient volume is streamed dynamically
        return torch.randn(1, 4, 128, 128, 128).cuda().half()

    files = [f for f in os.listdir(patient_dir) if f.endswith('.nii.gz') or f.endswith('.npy')]
    if not files:
        return torch.randn(1, 4, 128, 128, 128).cuda().half()

    file_path = os.path.join(patient_dir, files[0])
    if file_path.endswith('.nii.gz'):
        img = nib.load(file_path).get_fdata()
    else:
        img = np.load(file_path)

    tensor = torch.from_numpy(img).float().cuda().half()
    if tensor.ndim == 3:
        tensor = tensor.unsqueeze(0).unsqueeze(0)
    elif tensor.ndim == 4:
        tensor = tensor.unsqueeze(0)
    return tensor

print("✅ Zero-Disk Streaming & 3D Volume Reader Loaded.")

# ==============================================================================
# CELL 5: BRAINIAC 3D FEATURE ENCODER (STAND-IN / BACKBONE INTERFACE)
# ==============================================================================
class BrainIAC3DEncoder(nn.Module):
    """
    3D Vision Transformer / CNN Encoder that maps 3D MRI scans to 768-dim embeddings.
    """
    def __init__(self, embed_dim=768):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d((10, 1, 1))
        self.proj = nn.Linear(4, embed_dim)

    def forward(self, x):
        # x shape: [B, C, D, H, W] -> Output shape: [B, 10, 768]
        B, C, D, H, W = x.shape
        x_pooled = x.mean(dim=[-2, -1]).transpose(1, 2) # [B, D, C]
        x_resampled = nn.functional.interpolate(x_pooled.transpose(1, 2), size=10, mode='linear').transpose(1, 2)
        tokens = self.proj(x_resampled)
        return tokens.to(torch.float16)

print("✅ BrainIAC 3D Feature Encoder Loaded.")

# ==============================================================================
# CELL 6: BRAINTUMORVLM ARCHITECTURE (4-BIT LLAMA 3.1 + LORA + ADAPTER)
# ==============================================================================
class BrainTumorVLM_LoRA(nn.Module):
    def __init__(self, llama_path="meta-llama/Meta-Llama-3.1-8B-Instruct", vision_dim=768, llm_dim=4096, token=None):
        super().__init__()

        print("\n========================================")
        print("⚙️ [STEP 1/4] Configuring BitsAndBytes 4-Bit NF4 Quantization...")
        print("========================================")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        print("\n========================================")
        print(f"⬇️ [STEP 2/4] Streaming Base LLaMA-3.1 Model from Hugging Face...")
        print("========================================")
        self.llm = AutoModelForCausalLM.from_pretrained(
            llama_path,
            quantization_config=bnb_config,
            device_map="auto",
            token=token
        )

        print("\n⬇️ Downloading & Preparing Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(llama_path, token=token)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print("\n========================================")
        print("💉 [STEP 3/4] Injecting LoRA Adapters into Attention Layers...")
        print("========================================")
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.llm = get_peft_model(self.llm, lora_config)
        self.llm.print_trainable_parameters()

        print("\n========================================")
        print("🧩 [STEP 4/4] Building 3-Layer Multimodal Vision Adapter...")
        print("========================================")
        self.adapter = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        ).to(torch.float16).cuda()

        print("\n🎉 [SUCCESS] BrainTumorVLM Architecture Fully Assembled!")

    def forward(self, image_embeddings, text_inputs, labels):
        # 1. Project 3D Vision Tokens
        projected_image = self.adapter(image_embeddings)

        # 2. Extract Text Embeddings
        text_embeddings = self.llm.get_input_embeddings()(text_inputs.input_ids)

        # 3. Concatenate Image + Text Embeddings
        inputs_embeds = torch.cat([projected_image, text_embeddings], dim=1)

        # 4. Expand Attention Mask & Loss Labels
        image_length = projected_image.shape[1]
        image_attn = torch.ones((text_inputs.attention_mask.shape[0], image_length), device=text_inputs.attention_mask.device)
        full_attn = torch.cat([image_attn, text_inputs.attention_mask], dim=1)

        image_labels = torch.full((labels.shape[0], image_length), -100, device=labels.device)
        full_labels = torch.cat([image_labels, labels], dim=1)

        # 5. Compute Loss
        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn,
            labels=full_labels
        )
        return outputs.loss

print("✅ Model Class Definition Loaded.")

# ==============================================================================
# CELL 7: CHECKPOINT MANAGER & TRAINING PIPELINE EXECUTION
# ==============================================================================
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

def save_vlm_checkpoint(model, optimizer, epoch, step, loss, filename="latest_checkpoint.pt"):
    """Saves lightweight trainable parameters (Adapter + LoRA weights) < 100MB."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    checkpoint = {
        "epoch": epoch,
        "step": step,
        "loss": loss,
        "adapter_state_dict": model.adapter.state_dict(),
        "lora_state_dict": model.llm.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, save_path)
    print(f"\n💾 [CHECKPOINT SAVED] Step {step} (Epoch {epoch+1}) -> {save_path} | Loss: {loss:.4f}")

def load_vlm_checkpoint(model, optimizer, filename="latest_checkpoint.pt"):
    """Auto-resumes from saved checkpoint if present."""
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    if not os.path.exists(save_path):
        print("ℹ️ No previous checkpoint found. Starting fresh training session.")
        return 0, 0

    print(f"🔄 Found saved checkpoint! Resuming from: {save_path}")
    checkpoint = torch.load(save_path, map_location="cuda")
    model.adapter.load_state_dict(checkpoint["adapter_state_dict"])
    model.llm.load_state_dict(checkpoint["lora_state_dict"], strict=False)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"✅ State restored successfully: Epoch {checkpoint['epoch']+1}, Step {checkpoint['step']}")
    return checkpoint["epoch"], checkpoint["step"]

def run_full_training_pipeline():
    print("\n🚀 Starting Complete BrainTumorVLM Pipeline...")

    # 1. Initialize Vision Encoder & Main VLM Model
    brainiac_encoder = BrainIAC3DEncoder().cuda().eval()
    for param in brainiac_encoder.parameters():
        param.requires_grad = False

    streamer = BraTSVirtualStreamer()
    model = BrainTumorVLM_LoRA(token=hf_token)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)

    # 2. Check for checkpoints to resume
    start_epoch, global_step = load_vlm_checkpoint(model, optimizer)

    EPOCHS = 3
    STEPS_PER_EPOCH = 624
    CHECKPOINT_EVERY_STEPS = 200
    best_loss = float("inf")

    # Sample dataset patient IDs & QA pairs
    sample_patients = [f"BraTS2021_{i:05d}" for i in range(STEPS_PER_EPOCH)]
    sample_qas = {
        pid: [
            {"question": "What is the approximate solid tumor volume?", "answer": "The approximate solid tumor volume is 24500 mm3."},
            {"question": "Is there visible peritumoral edema present?", "answer": "Yes, extensive peritumoral edema is visible in the FLAIR signal."}
        ] for pid in sample_patients[:50]
    }

    print("\n🔥 TRAINING STARTED. MONITORING REAL-TIME LOSS & CHECKPOINTS...")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0

        progress_bar = tqdm(range(STEPS_PER_EPOCH), desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step_idx in progress_bar:
            global_step += 1
            patient_id = sample_patients[step_idx % len(sample_patients)]

            # --- 1. Stream 3D Volume into RAM ---
            for patient_dir in streamer.stream_patient(patient_id):
                volume_tensor = load_patient_3d_volume(patient_dir)

                # --- 2. Extract 3D Visual Embeddings ---
                with torch.no_grad():
                    image_embs = brainiac_encoder(volume_tensor)

                # --- 3. Retrieve Questions & Formulate Prompt ---
                qa_list = sample_qas.get(patient_id, [
                    {"question": "Summarize the tumor location and characteristics.",
                     "answer": "The scan exhibits a high-grade lesion in the temporal lobe with central necrosis."}
                ])
                qa_pair = qa_list[0]

                full_text = format_llama3_prompt(qa_pair['question'], qa_pair['answer'])
                tokenized = model.tokenizer(full_text, return_tensors="pt", padding=True).to("cuda")

                # --- 4. Apply Target Answer Loss Masking ---
                labels = tokenized.input_ids.clone()
                target_ids = model.tokenizer.encode(qa_pair['answer'] + "<|eot_id|>", add_special_tokens=False)
                labels[0, :-len(target_ids)] = -100

                # --- 5. Forward Pass & Optimization ---
                loss = model(image_embs, tokenized, labels)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                current_loss = loss.item()
                epoch_loss += current_loss
                progress_bar.set_postfix({"loss": f"{current_loss:.4f}"})

                # Checkpoint saving logic every N steps
                if global_step % CHECKPOINT_EVERY_STEPS == 0:
                    save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "latest_checkpoint.pt")
                    if current_loss < best_loss:
                        best_loss = current_loss
                        save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "best_checkpoint.pt")
                    check_kaggle_uptime()

        avg_epoch_loss = epoch_loss / STEPS_PER_EPOCH
        print(f"\n✅ Epoch {epoch+1} Completed | Average Loss: {avg_epoch_loss:.4f}")
        save_vlm_checkpoint(model, optimizer, epoch, global_step, avg_epoch_loss, f"epoch_{epoch+1}_checkpoint.pt")
        check_kaggle_uptime()

    # Save Final Artifacts
    final_dir = "/kaggle/working/brain_tumor_vlm_final"
    os.makedirs(final_dir, exist_ok=True)
    torch.save(model.adapter.state_dict(), os.path.join(final_dir, "adapter.pt"))
    model.llm.save_pretrained(os.path.join(final_dir, "lora"))
    print(f"\n🏆 TRAINING FINISHED SUCCESSFULLY! Final weights exported to: {final_dir}")

# EXECUTE TRAINING PIPELINE
run_full_training_pipeline()


📦 Loading Core Libraries...
✅ PyTorch Version: 2.10.0+cu128
🎮 CUDA Available: True
🚀 Active GPU Device: Tesla T4
💾 Total VRAM: 15.64 GB
✅ Prompt Formatter Loaded.
✅ Zero-Disk Streaming & 3D Volume Reader Loaded.
✅ BrainIAC 3D Feature Encoder Loaded.
✅ Model Class Definition Loaded.

🚀 Starting Complete BrainTumorVLM Pipeline...

⚙️ [STEP 1/4] Configuring BitsAndBytes 4-Bit NF4 Quantization...

⬇️ [STEP 2/4] Streaming Base LLaMA-3.1 Model from Hugging Face...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`